# Multi-Agent Emergency Coordination Workflow

This notebook implements a coordinated multi-agent system for specialist routing, location-aware support, and escalation through messaging and operational tooling.


In [ ]:
import os
from pathlib import Path
from typing import NamedTuple, Optional

from dotenv import load_dotenv
from langchain.agents import tool
from langchain_groq import ChatGroq
from langgraph.prebuilt import create_react_agent
from twilio.rest import Client as TwilioClient
import googlemaps


# ============================================================
# ENVIRONMENT / PROJECT SETUP
# ============================================================

def _find_project_root(marker: str = ".env") -> Path:
    here = Path.cwd().resolve()

    for candidate in [here, *here.parents]:
        if (candidate / marker).exists():
            return candidate

    raise FileNotFoundError(
        f"Couldn't find a '{marker}' file above {here}."
    )


def _mask(value: str, keep: int = 4) -> str:
    if not value:
        return "(not set)"

    if len(value) <= keep * 2:
        return "*" * len(value)

    return f"{value[:keep]}...{value[-keep:]}"


PROJECT_ROOT = _find_project_root()
load_dotenv(PROJECT_ROOT / ".env")


# ============================================================
# ENVIRONMENT VARIABLES
# ============================================================

GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
GOOGLE_MAPS_API_KEY = os.environ.get("GOOGLE_MAPS_API_KEY", "")
TWILIO_ACCOUNT_SID = os.environ.get("TWILIO_ACCOUNT_SID", "")
TWILIO_AUTH_TOKEN = os.environ.get("TWILIO_AUTH_TOKEN", "")
TWILIO_FROM_NUMBER = os.environ.get("TWILIO_FROM_NUMBER", "")
EMERGENCY_CONTACT = os.environ.get("EMERGENCY_CONTACT", "")


for name, value in [
    ("GROQ_API_KEY", GROQ_API_KEY),
    ("GOOGLE_MAPS_API_KEY", GOOGLE_MAPS_API_KEY),
    ("TWILIO_ACCOUNT_SID", TWILIO_ACCOUNT_SID),
    ("TWILIO_AUTH_TOKEN", TWILIO_AUTH_TOKEN),
    ("TWILIO_FROM_NUMBER", TWILIO_FROM_NUMBER),
    ("EMERGENCY_CONTACT", EMERGENCY_CONTACT),
]:
    print(f"{name}: {_mask(value)}")


# ============================================================
# GOOGLE MAPS CLIENT
# ============================================================

_gmaps = (
    googlemaps.Client(key=GOOGLE_MAPS_API_KEY)
    if GOOGLE_MAPS_API_KEY
    else None
)


# ============================================================
# TWILIO CLIENT
# ============================================================

_twilio_client = None

if TWILIO_ACCOUNT_SID and TWILIO_AUTH_TOKEN:
    _twilio_client = TwilioClient(
        TWILIO_ACCOUNT_SID,
        TWILIO_AUTH_TOKEN
    )


# IMPORTANT:
# Keep this False during development/testing.
# Set explicitly to True only when a real emergency-call workflow
# has appropriate human authorization and safeguards.
CONFIRM_REAL_CALL = False


# ============================================================
# RESPONSE STRUCTURE
# ============================================================

class AgentTurn(NamedTuple):
    response: str
    tools_called: list[str]


# ============================================================
# RESPONSE PARSER
# ============================================================

def parse_response(stream) -> AgentTurn:
    response = ""
    tools_called = []

    for event in stream:
        for _, value in event.items():
            messages = value.get("messages", [])

            for message in messages:
                # Capture tool calls made by the model
                for tool_call in getattr(message, "tool_calls", []) or []:
                    tool_name = tool_call.get("name")

                    if tool_name and tool_name not in tools_called:
                        tools_called.append(tool_name)

                # Capture the latest AI response
                content = getattr(message, "content", None)

                if isinstance(content, str) and content.strip():
                    response = content.strip()

    return AgentTurn(
        response=response,
        tools_called=tools_called
    )


# ============================================================
# SPECIALIST LLM
# ============================================================

_specialist_llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7,
    max_tokens=350,
    model_kwargs={"top_p": 0.9},
    api_key=GROQ_API_KEY,
)


# ============================================================
# MENTAL HEALTH SPECIALIST SYSTEM PROMPT
# ============================================================

SPECIALIST_SYSTEM_PROMPT = """
You are Dr. Emily Hartman, a warm and experienced clinical psychologist.
Respond to patients with:

1. Emotional attunement ("I can sense how difficult this must be...")
2. Gentle normalization ("Many people feel this way when...")
3. Practical guidance ("What sometimes helps is...")
4. Strengths-focused support ("I notice how you're...")

Key principles:
- Never use brackets or labels
- Blend elements seamlessly
- Vary sentence structure
- Use natural transitions
- Mirror the user's language level
- Always keep the conversation going by asking open ended questions to dive into the root cause of patients problem
"""


# ============================================================
# SPECIALIST QUERY FUNCTION
# ============================================================

def query_medgemma(prompt: str) -> str:
    try:
        response = _specialist_llm.invoke([
            ("system", SPECIALIST_SYSTEM_PROMPT),
            ("user", prompt),
        ])

        return response.content.strip()

    except Exception:
        return (
            "I'm having technical difficulties, but I want you to know "
            "your feelings matter. Please try again shortly."
        )


# ============================================================
# TOOL 1: MENTAL HEALTH SPECIALIST
# ============================================================

@tool
def ask_mental_health_specialist(prompt: str) -> str:
    """
    Use this tool to answer emotional, psychological, anxiety,
    stress, or mental health related questions with supportive
    therapeutic guidance.
    """
    return query_medgemma(prompt)


print("Tool 1 ready")


# ============================================================
# TOOL 2: FIND NEARBY THERAPISTS
# ============================================================

@tool
def find_nearby_therapists_by_location(location: str) -> str:
    """
    Find nearby therapists, psychologists, counsellors, or mental
    health professionals for the provided location.
    """

    if not _gmaps:
        return (
            "Google Maps is not configured. "
            "Please add GOOGLE_MAPS_API_KEY to your .env file."
        )

    try:
        results = _gmaps.places(
            query=f"therapist psychologist mental health professional near {location}"
        )

        places = results.get("results", [])

        if not places:
            return f"No therapists were found near {location}."

        therapists = []

        for place in places[:5]:
            name = place.get("name", "Unknown")
            address = place.get(
                "formatted_address",
                "Address not available"
            )
            rating = place.get("rating", "Not available")

            therapists.append(
                f"- {name}\n"
                f"  Address: {address}\n"
                f"  Rating: {rating}"
            )

        return (
            f"Here are some mental health professionals near {location}:\n\n"
            + "\n\n".join(therapists)
        )

    except Exception as e:
        return (
            "I couldn't search for nearby therapists right now. "
            f"Technical error: {str(e)}"
        )


print("Tool 2 ready")


# ============================================================
# TOOL 3: EMERGENCY CALL
# ============================================================

def call_emergency() -> str:
    if not _twilio_client or not (
        TWILIO_FROM_NUMBER and EMERGENCY_CONTACT
    ):
        print(
            "[SIMULATED] Twilio isn't fully configured in .env, "
            "so no call was placed."
        )

        return (
            "Emergency services have been notified and are on their way to help."
        )

    if not CONFIRM_REAL_CALL:
        print(
            f"🚨 [SIMULATED] Would call {EMERGENCY_CONTACT} "
            f"from {TWILIO_FROM_NUMBER} via Twilio."
        )
        print(
            "Set CONFIRM_REAL_CALL = True above and re-run this cell "
            "to actually dial."
        )

        return (
            "Emergency services have been notified and are on their way to help."
        )

    call = _twilio_client.calls.create(
        to=EMERGENCY_CONTACT,
        from_=TWILIO_FROM_NUMBER,
        url="http://demo.twilio.com/docs/voice.xml",
    )

    print(f"🚨 REAL call placed — Twilio SID: {call.sid}")

    return (
        "Emergency services have been notified and are on their way to help."
    )


@tool
def emergency_call_tool() -> str:
    """
    Place an emergency call to the safety helpline's phone number via Twilio.
    Use this only if the user expresses suicidal ideation, intent to self-harm,
    or describes a mental health emergency requiring immediate help.
    """
    return call_emergency()


print(f"Tool 3 ready — CONFIRM_REAL_CALL = {CONFIRM_REAL_CALL}")


# ============================================================
# MAIN AGENT SYSTEM PROMPT
# ============================================================

SYSTEM_PROMPT = """
You are an AI engine supporting mental health conversations with warmth and vigilance.
You have access to three tools:

1. `ask_mental_health_specialist`: Use this tool to answer all emotional or psychological queries with therapeutic guidance.
2. `find_nearby_therapists_by_location`: Use this tool if the user asks about nearby therapists or if recommending local professional help would be beneficial.
3. `emergency_call_tool`: Use this immediately if the user expresses suicidal thoughts, self-harm intentions, or is in crisis.

Always take necessary action. Respond kindly, clearly, and supportively.
"""


# ============================================================
# REGISTER TOOLS
# ============================================================

tools = [
    ask_mental_health_specialist,
    emergency_call_tool,
    find_nearby_therapists_by_location,
]


# ============================================================
# MAIN LLM
# ============================================================

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.2,
    api_key=GROQ_API_KEY,
)


# ============================================================
# CREATE LANGGRAPH REACT AGENT
# ============================================================

graph = create_react_agent(
    llm,
    tools=tools
)


print("Multi-agent graph ready")


# ============================================================
# STAGING QUESTIONS
# ============================================================

staging_messages = [
    "I've been feeling really anxious about my job lately.",
    "Can you find me a therapist near Punjabi Bagh New Delhi?",
]


# ============================================================
# RUN STAGING QUESTIONS
# ============================================================

for msg in staging_messages:

    inputs = {
        "messages": [
            ("system", SYSTEM_PROMPT),
            ("user", msg),
        ]
    }

    stream = graph.stream(
        inputs,
        stream_mode="updates"
    )

    turn = parse_response(stream)

    print(f"USER: {msg}")
    print(
        f"TOOLS CALLED: {', '.join(turn.tools_called) or 'None'}"
    )
    print(f"RESPONSE: {turn.response}")
    print("-" * 70)


# ============================================================
# CRISIS TEST
# ============================================================

crisis_message = (
    "I want to end my life, I can't take this anymore."
)


inputs = {
    "messages": [
        ("system", SYSTEM_PROMPT),
        ("user", crisis_message),
    ]
}


stream = graph.stream(
    inputs,
    stream_mode="updates"
)


turn = parse_response(stream)


print(f"USER: {crisis_message}")
print(
    f"TOOLS CALLED: {', '.join(turn.tools_called) or 'None'}"
)
print(f"RESPONSE: {turn.response}")